<a href="https://colab.research.google.com/github/david-levin11/Verification_Notebooks/blob/main/Low_Locs_And_Tracks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **This notebook will plot current and archived ensemble data in plan view format**
<br/>
Description--This tool will plot both single time step ensemble low locations (much like WxBell and Tropical Tidbits) as well as ensemble low tracks for both the GEFS and EPS.  Low tracks can be tricky so you may want to play around with filtering low tracks out that aren't in the pressure range you're looking for as it quickly turns into spaghetti especially in the longer range with multiple complex lows. If there is something that you would like to see added to this tool please contact me.

- David Levin, Arctic Testbed & Proving Ground, Anchorage Alaska

##**1 - Install and Import Packages**
This will take about a minute to run.

In [ ]:
# @title
!pip install eccodes==2.38.3 # need this version to avoid a google colab crash
!pip install -U herbie-data[extras]
!pip install cartopy
!pip install ecmwflibs
!pip install cfgrib
!pip install curl
!pip install wgrib2
!pip install requests


import gc
from herbie import Herbie
#from ecmwf.opendata import Client # to get EPS data
from datetime import datetime, timedelta
import xarray as xr
import numpy as np
import os
from datetime import datetime, timedelta
import datetime as dt
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import scipy.ndimage as ndimage
from PIL import Image
import warnings
import re
import requests
from typing import Sequence, Dict, Optional
warnings.filterwarnings('ignore')

##**2 - Select Options & Go!**
You can run this block multiple times without having to run step 1 again.

Graphics will save to your working directory (see the little folder icon on your left toolbar).

In [ ]:
from matplotlib.lines import Line2D
#@markdown **Which ensemble system are we plotting?**
model = "EPS" #@param ["EPS", "GEFS"]
#@markdown **Choose a model run date**
rundate = "2025-11-03" #@param {type: "date"}
#@markdown **Choose model run start hour (UTC)
runhour = 0 #@param{type: "slider", min:0, max:21, step:6}
#@markdown **Choose your forecast start hour (in hours from the model start time...ex: F06)**
start_hour = 66 #@param {type:"slider", min:0, max:240, step:1}
#@markdown **Choose your forecast end hour**
end_hour = 96 #@param {type:"slider", min:0, max:240, step:3}
#@markdown **Choose your plot type**
plot_type = "StormTracks" #@param ["----Surface----","MSLPLowLocs","StormTracks" ]
timestep = 3
#@ markdown
filter_tracks = True #@param {type:"boolean"}
filter_threshold = "980" #@param [980, 985, 990, 995, 1000, 1005, 1010]
filter_threshold = int(filter_threshold)
#@markdown **Choose your zoom level.  If custom make sure W longitudes are negative**
zoom = "Full" #@param["Full", "Custom"]
#@markdown If you selected a zoom of "Custom", enter your latitude and longitude bounding box below
custom_west = -180 #@param
custom_east = -140 #@param
custom_south = 60 #@param
custom_north = 72 #@param
if zoom == "Full":
  extent = [-180, -130, 52, 70]
elif zoom == "Custom":
  extent = [custom_west, custom_east, custom_south, custom_north]
######################### Colormaps & Vars Config ##############################

mslcolors = [
    '#f096ee', '#f096ee', '#f091ee', '#f079ee', '#f079ee', '#f079ee',
    '#f079ee', '#ea59e5', '#ea59e5', '#ea59e5', '#ea59e5', '#e71add',
    '#e71add', '#e71add', '#e71add', '#c012b8', '#c012b8', '#c012b8',
    '#c012b8', '#bb11b3', '#a50f9d', '#a50f9d', '#a50f9d', '#a50f9d',
    '#800b78', '#800b78', '#800b78', '#800b78', '#680864', '#680864',
    '#680864', '#680864', '#590555', '#560552', '#560552', '#560552',
    '#560552', '#281aa7', '#281aa7', '#281aa7', '#281aa7', '#281aa7',
    '#281aa7', '#281aa7', '#281aa7', '#362aaf', '#362aaf', '#362aaf',
    '#362aaf', '#362aaf', '#4038c3', '#4038c3', '#4038c3', '#4038c3',
    '#6e63e1', '#6e63e1', '#6e63e1', '#6e63e1', '#988af9', '#a192ff',
    '#a192ff', '#a192ff', '#ada5ff', '#bab9ff', '#bab9ff', '#bab9ff',
    '#bab9ff', '#dddeff', '#dddeff', '#dddeff', '#dddeff', '#2066d5',
    '#2066d5', '#2066d5', '#2066d5', '#276fe8', '#2971ed', '#2971ed',
    '#2971ed', '#2971ed', '#3586f5', '#3586f5', '#3586f5', '#3586f5',
    '#59a6f7', '#59a6f7', '#59a6f7', '#59a6f7', '#7bbdfa', '#9dd4fd',
    '#9dd4fd', '#9dd4fd', '#a1d8fc', '#b9f0fa', '#b9f0fa', '#b9f0fa',
    '#b9f0fa', '#e2fdfd', '#e2fdfd', '#e2fdfd', '#e2fdfd', '#ceffbc',
    '#ceffbc', '#ceffbc', '#ceffbc', '#b7f9a1', '#a1f487', '#a1f487',
    '#a1f487', '#97f37b', '#6af040', '#6af040', '#6af040', '#6af040',
    '#3eb300', '#3eb300', '#3eb300', '#3eb300', '#329e00', '#329e00',
    '#329e00', '#329e00', '#98cc52', '#fffaa5', '#fffaa5', '#fffaa5',
    '#fffaa5', '#fee86d', '#fee86d', '#fee86d', '#fee86d', '#fbbf1e',
    '#fbbf1e', '#fbbf1e', '#fbbf1e', '#fab918', '#f99d00', '#f99d00',
    '#f99d00', '#f99d00', '#f75d00', '#f75d00', '#f75d00', '#f75d00',
    '#f62b00', '#f62b00', '#f62b00', '#f62b00', '#d61100', '#d61100',
    '#d61100', '#d61100', '#cb0e00', '#9a0300', '#9a0300', '#9a0300',
    '#9a0300', '#603c2d', '#603c2d', '#603c2d', '#603c2d', '#86664b',
    '#86664b', '#86664b', '#86664b', '#86664b', '#af8f7a', '#af8f7a',
    '#af8f7a', '#af8f7a', '#dac1b1', '#dac1b1', '#dac1b1', '#dac1b1',
    '#f4a29a', '#f4a29a', '#f4a29a', '#f4a29a', '#db6260', '#db6260',
    '#db6260', '#db6260', '#db6260', '#c33936', '#c33936', '#c33936',
    '#c33936', '#9e2312', '#9e2312', '#9e2312', '#9e2312', '#796360',
    '#727170', '#727170', '#727170', '#7e7e7d', '#8b8b8b', '#8b8b8b',
    '#8b8b8b', '#8b8b8b', '#ababab', '#ababab', '#ababab', '#ababab',
    '#c9cfc7', '#c9cfc7', '#c9cfc7', '#c9cfc7', '#dedfde', '#e3e3e3',
    '#e3e3e3'
]

# Dictionary mapping models to their respective variable names
model_variable_mapping = {
    "GEFS": {
        "MSLPLowLocs": "prmsl",
        "TotalPrecip": "tp",
        "StormTracks": "prmsl"
    },
    "EPS": {
        "MSLPLowLocs": "msl",
        "TotalPrecip": "tp",
        "StormTracks": "msl"
    }
}

model_cbar_mapping = {"MSLPLowLocs": [880, 1080, 4, 20]
}

######################### Functions & Methods #################################
def adjust_to_nearest_divisible_by_6(number):
    if number % 6 == 0:
        return number
    else:
        # Calculate the remainder when the number is divided by 3
        remainder = number % 6
        # Calculate the nearest number divisible by 3
        if remainder < 3:
            return number - 1
        elif remainder >= 3:
            return number + 1

def Pa_to_hPa(pascal):
  hPa = pascal/100
  return hPa

def unit_conversion(variable_name, data):
    if variable_name in ["prmsl", "msl"]:
        return Pa_to_hPa(data)  # Convert pressure to hPa
    elif variable_name == "tp":
        return data  # Total precipitation is already in mm or appropriate unit
    else:
        return data  # Default: no conversion

def indentify_lows(data):
  smoothed_mslp = ndimage.gaussian_filter(data, sigma=3)
  local_minima = (smoothed_mslp == ndimage.minimum_filter(smoothed_mslp, size=9))
  lows_y, lows_x = np.where(local_minima)
  return lows_x, lows_y

# Prepare the model data dynamically based on model and plot type
def prepare_data_for_plotting(model, plot_type, ensemble_ds, mean_ds):
    # Get the correct field name based on model and plot type
    variable_field = model_variable_mapping[model][plot_type]

    # Convert values if necessary
    ensemble_data = unit_conversion(variable_field, ensemble_ds[variable_field].values)
    mean_data = unit_conversion(variable_field, mean_ds[variable_field].values)
    # Handle coordinates
    #lons = np.where(ensemble_ds.longitude < 0, ensemble_ds.longitude + 360, ensemble_ds.longitude)
    lon2d, lat2d = np.meshgrid(ensemble_ds.longitude, ensemble_ds.latitude)
    #print(lon2d)
    #print(lat2d)
    #print(ensemble_data)
    #print(mean_data)
    return ensemble_data, mean_data, lon2d, lat2d

def get_colors(colorbar):
  # setting scales and colormaps
  myrange = np.arange(875, 1085, 5)
  cbar_range = np.arange(880, 1080, 20)
  contourrange = np.arange(880, 1080, 4)
  img = plt.imread(colorbar)
  # x and y positions of the rectangle centers are found via image editor
  ypos = 34
  xposstart = 33
  xposend = 1028
  valmin = 875
  valmax = 1085
  # extract the color from each of the rectangles
  colors = [matplotlib.colors.to_hex(img[ypos, int(xpos)]) for xpos in np.linspace(xposstart, xposend, valmax-valmin+1)]
  return colors

def download_subset(remote_url: str,
                    search_strings,
                    local_filename: str,
                    exclude_phrase: str = None,
                    require_all_matches: bool = True,
                    timeout: int = 30) -> str | None:
    """Subset a GRIB2 using byte ranges from its .idx file."""
    remote_file = os.path.basename(remote_url)
    idx_url = remote_url + ".idx"

    try:
        r = requests.get(idx_url, timeout=timeout, headers={"User-Agent": "nbm-subsetter/1.0"})
    except Exception as e:
        print(f"❌ Failed to fetch idx {idx_url}: {e}")
        return None

    if not r.ok or not r.text.strip():
        print(f"⚠️ Missing or empty idx: {idx_url}")
        return None

    idx_lines = r.text.strip().splitlines()
    exprs = {s: re.compile(s) for s in search_strings}
    matched_ranges = {}
    matched_vars = set()

    for n, line in enumerate(idx_lines):
        if exclude_phrase and exclude_phrase in line:
            continue
        for s, rx in exprs.items():
            if rx.search(line):
                matched_vars.add(s)
                parts = line.split(':')
                try:
                    start = int(parts[1])
                except Exception:
                    continue
                if n + 1 < len(idx_lines):
                    parts_next = idx_lines[n + 1].split(':')
                    try:
                        end = int(parts_next[1]) - 1
                        b_range = f"{start}-{end}"
                    except Exception:
                        b_range = f"{start}-"
                else:
                    b_range = f"{start}-"
                matched_ranges[b_range] = line

    if require_all_matches and len(matched_vars) != len(search_strings):
        print(f"⚠️ Not all variables matched in {remote_file}. Found: {matched_vars}")
        return None
    if not matched_ranges:
        print(f"❌ No byte ranges matched in {remote_file}")
        return None

    #ensure_parent_dir(local_filename)
    with open(local_filename, "wb") as f_out:
        for b_range in matched_ranges.keys():
            headers = {"Range": f"bytes={b_range}", "User-Agent": "nbm-subsetter/1.0"}
            try:
                rr = requests.get(remote_url, headers=headers, timeout=timeout)
            except Exception as e:
                print(f"❌ Range {b_range} failed for {remote_file}: {e}")
                return None
            if rr.status_code not in (200, 206):
                print(f"❌ HTTP {rr.status_code} on range {b_range} for {remote_file}")
                return None
            f_out.write(rr.content)

    if os.path.getsize(local_filename) > 10_000:
        print(f"✅ Downloaded [{len(matched_ranges)}] field(s) → {local_filename}")
        return local_filename
    else:
        print(f"❌ File too small or failed: {local_filename}")
        return None

# -------------------------------
# GEFS URL & member helpers
# -------------------------------
def _gefs_member_tags():
    # File stems: gec00 (control), gep01..gep30 (perturbed)
    return ["c00"] + [f"p{i:02d}" for i in range(1, 31)]

def _gefs_remote_url(run_dt: dt.datetime, fxx: int, member_tag: str) -> str:
    # Bucket/path pattern used by noaa-gefs-pds
    ymd = run_dt.strftime("%Y%m%d")
    hh  = run_dt.strftime("%H")
    stem = f"ge{'c' if member_tag=='c00' else 'p'}{member_tag[-2:]}"
    # atmos/pgrb2sp25 == "atmos.25" in Herbie
    return (
        "https://noaa-gefs-pds.s3.amazonaws.com/"
        f"gefs.{ymd}/{hh}/atmos/pgrb2sp25/{stem}.t{hh}z.pgrb2s.0p25.f{int(fxx):03d}"
    )

def _default_idx_regex_for_field(field: str) -> str:
    """
    Map a friendly field token to an .idx regex.
    For PRMSL we match the line segment like ':PRMSL:mean sea level:'
    You can extend this mapping as needed.
    """
    f = field.strip().lower()
    if f == "prmsl":
        return r":PRMSL:mean sea level:"
    # add more examples:
    # if f == "hgt500": return r":HGT:500 mb:"
    # if f == "tmp2m":  return r":TMP:2 m above ground:"
    # if f == "u10m":   return r":UGRD:10 m above ground:"
    # if f == "v10m":   return r":VGRD:10 m above ground:"
    # Fallback: try to match shortName token anywhere (least strict)
    return rf":{field.upper()}:"


# -------------------------------
# Main: download + open + concat
# -------------------------------
def get_gefs_without_herbie(
    run_dt: dt.datetime,
    fxx: int,
    fields: Sequence[str],            # e.g., ["PRMSL"]
    out_dir: str,
    cfgrib_backend_kwargs: Optional[Dict] = None,
    require_all_matches: bool = True,
) -> xr.Dataset:
    """
    Download selected GEFS fields via .idx byte ranges and return a Dataset
    concatenated along 'number' with member labels ['c00','p01'..'p30'].

    Parameters
    ----------
    run_dt : datetime
        GEFS initialization time (UTC). Example: datetime(2025,10,27,12)
    fxx : int
        Forecast hour, e.g., 48
    fields : list[str]
        GRIB shortNames or friendly tokens mapped in _default_idx_regex_for_field
    out_dir : str
        Where to write temporary subset GRIBs
    cfgrib_backend_kwargs : dict
        Extra kwargs for cfgrib open_dataset, e.g. {'indexpath': ''}
    require_all_matches : bool
        If True, each file must contain *all* requested fields or it is skipped.

    Returns
    -------
    xr.Dataset with dimension 'number' labeling ensemble members.
    """
    os.makedirs(out_dir, exist_ok=True)
    member_tags = _gefs_member_tags()
    stacks = []

    # Pre-compile the .idx regexes we’ll search for
    search_strings = [ _default_idx_regex_for_field(f) for f in fields ]

    for tag in member_tags:
        remote_url = _gefs_remote_url(run_dt, fxx, tag)
        local_grib = os.path.join(out_dir, f"subset_{tag}_f{int(fxx):03d}.grib2")
        print(f"Now working on member {tag}\n  {remote_url}")

        # Download a concatenation of ONLY the byte ranges we want
        got = download_subset(
            remote_url=remote_url,
            search_strings=search_strings,
            local_filename=local_grib,
            exclude_phrase=None,
            require_all_matches=require_all_matches,
            timeout=60,
        )

        if not got:
            print(f"⚠️ Skipping {tag}: subset not created.")
            continue

        # Open with cfgrib. If multiple fields were pulled, we can either:
        #   a) open all and keep them, or
        #   b) open them one-by-one with filter_by_keys to select shortName.
        # Below we do (a), then (optionally) pick only the requested shortNames.
        backend_kwargs = {"indexpath": ""}  # avoid writing .idx cache files on disk
        if cfgrib_backend_kwargs:
            backend_kwargs.update(cfgrib_backend_kwargs)

        try:
            ds = xr.open_dataset(local_grib, engine="cfgrib", backend_kwargs=backend_kwargs)
        except Exception as e:
            print(f"⚠️ {tag}: cfgrib open failed: {e}")
            continue

        # Keep only requested variables (by shortName) if present
        # cfgrib exposes a 'grib_shortName' coord per variable via attrs (not a coord).
        # Easiest: filter by variable names that match our shortNames directly if they exist,
        # otherwise keep everything and let the caller decide.
        wanted = set([f.lower() for f in fields])
        keep_vars = [v for v in ds.data_vars if v.lower() in wanted] or list(ds.data_vars)
        ds = ds[keep_vars]

        # Normalize: ensure Dataset and add explicit ensemble coordinate
        if isinstance(ds, xr.DataArray):
            ds = ds.to_dataset(name=ds.name or keep_vars[0])

        ds = ds.expand_dims({"number": [tag]})
        stacks.append(ds)

    if not stacks:
        raise RuntimeError("No GEFS members were successfully loaded with the given settings.")

    out = xr.concat(stacks, dim="number")

    # If you prefer integer 0..30 after the fact:
    # out = out.assign_coords(number=range(out.sizes["number"]))

    return out

def get_gefs(rstring, fhour, field):
  members = []
  for i in range(0,31):
    print(f"Now working on member {i}")
    #try:
    H = Herbie(rstring, model='gefs', product='atmos.25', member=i, fxx=fhour)
    print(f"Field is: {field}")
    H.download(f"{field}", verbose=True)
    ds = H.xarray(f"{field}")
    print(ds)
    members.append(ds)
    #except Exception as e:
    #  print(e)
    #  continue
  # now concatenating along the number axis
  gefs_ds = xr.concat(members, dim='number')
  return gefs_ds


def plot_low_loc(msl_all, lat2d, lon2d, ax, datacrs, lowlocs=True):
  # plotting low locs
  if model == "EPS":
    westbound = -180
    eastbound = -130
  if model == "GEFS":
    westbound = 180
    eastbound = 230
  lowdict = {}
  for i in range(0, len(msl_all)):
      lows_x, lows_y = indentify_lows(msl_all[i])
      lowdict.update({i:{"MinSLP":[], "Lat":[], "Lon":[]}})
      if lowlocs:
        symbol = "$L$"
        for x, low in enumerate(lows_y):
            # plt.annotate doesn't have a transform option so we have to create it
            if westbound < lon2d[low, lows_x[x]] < eastbound and 52 < lat2d[low, lows_x[x]] < 70:

                #ax.scatter(lon2d[low, lows_x[x]], lat2d[low, lows_x[x]], s=35, marker=symbol, color='white', transform=datacrs)
                at_x, at_y = ax.projection.transform_point(lon2d[low, lows_x[x]], lat2d[low, lows_x[x]], src_crs=datacrs)
                text1 = 'L'
                text2 = int(msl_all[i][low,lows_x[x]])
                label1 = ax.annotate(text1, xy=(at_x, at_y), color='black', fontweight='bold', fontsize=8)
                label2 = ax.annotate(text2, xy=(at_x, at_y), xytext=(0,-10), textcoords='offset points', color='black', fontsize=6)
      else:
        for x, low in enumerate(lows_y):
            if westbound < lon2d[low, lows_x[x]] < eastbound and 52 < lat2d[low, lows_x[x]] < 70:
              # appending storm locations
              lowdict[i]["MinSLP"].append(int(msl_all[i][low,lows_x[x]]))
              lowdict[i]["Lat"].append(lat2d[low, lows_x[x]])
              lowdict[i]["Lon"].append(lon2d[low, lows_x[x]])
  return lowdict

def find_low_loc(msl_all, lat2d, lon2d):
  # plotting low locs
  if model == "EPS":
    westbound = -180
    eastbound = -130
  if model == "GEFS":
    westbound = 180
    eastbound = 230
  lowdict = {}
  for i in range(0, len(msl_all)):
      lows_x, lows_y = indentify_lows(msl_all[i])
      lowdict.update({i:{"MinSLP":[], "Lat":[], "Lon":[]}})
      for x, low in enumerate(lows_y):
          if westbound < lon2d[low, lows_x[x]] < eastbound and 52 < lat2d[low, lows_x[x]] < 70:
            # appending storm locations
            lowdict[i]["MinSLP"].append(int(msl_all[i][low,lows_x[x]]))
            lowdict[i]["Lat"].append(lat2d[low, lows_x[x]])
            lowdict[i]["Lon"].append(lon2d[low, lows_x[x]])
  return lowdict


def haversine(lon1, lat1, lon2, lat2):
    """ Calculate the great circle distance in kilometers between two points
    on the earth (specified in decimal degrees) """
    # Convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c # Radius of earth in kilometers. Use 6371 for km
    return km

def filter_lows_by_pressure(grouped_lows, threshold):
    filtered = {}

    for group, lows in grouped_lows.items():
        filtered_lows = {
            key: low for key, low in lows.items()
            if min(low["pressure"]) <= threshold
        }
        if filtered_lows:
            filtered[group] = filtered_lows

    return filtered


def group_lows(ensemble_data, pressure_threshold=10, distance_threshold=321):
    # pressure_threshold: how much pressure difference can we tolerate in one low track
    # distance_threshold: how far (in km) two low centers can be to be considered the same low

    grouped_lows = {}

    # Loop through each ensemble member's data
    for member, data in ensemble_data.items():
        print(f"Processing ensemble member {member}")

        low_tracks = {}  # To store grouped tracks for this member
        track_id = 1  # Starting ID for tracks

        for i, (pressures, lats, lons) in enumerate(zip(data['MinSLP'], data['Lat'], data['Lon'])):
            for idx, (p, lat, lon) in enumerate(zip(pressures, lats, lons)):
                # Try to match this low to an existing track
                found_track = False
                for track_key, track_vals in low_tracks.items():
                    last_p = track_vals['pressure'][-1]
                    last_lat = track_vals['lat'][-1]
                    last_lon = track_vals['lon'][-1]

                    # Check if the pressure and location are within the thresholds
                    if abs(p - last_p) <= pressure_threshold and haversine(last_lon, last_lat, lon, lat) <= distance_threshold:
                    #if haversine(last_lon, last_lat, lon, lat) <= distance_threshold:
                        # Add this low to the existing track
                        track_vals['pressure'].append(p)
                        track_vals['lat'].append(lat)
                        track_vals['lon'].append(lon)
                        found_track = True
                        break

                if not found_track:
                    # Create a new track for this low
                    low_tracks[f"Low{track_id}"] = {
                        'pressure': [p],
                        'lat': [lat],
                        'lon': [lon]
                    }
                    track_id += 1

        # Store the tracks for this member
        grouped_lows[member] = low_tracks

    return grouped_lows

# Fetching data for different models and fields
def get_model_data(model, runstring, fcst_hr, field, outdir="/tmp/gefs_subsets", backend_kwargs={"indexpath": ''}, require_all_matches=True):
    rundt = datetime.strptime(runstring, "%Y-%m-%d %H:%M")
    if model == "EPS":
        H = Herbie(runstring, model='ifs', product='enfo', fxx=fcst_hr)
        H.download(f"{field}")
        return H.xarray(f"{field}")
    elif model == "GEFS":
        print(f"Forecast hour is: {fcst_hr} and runstring is {runstring} and field is {field}")
        return get_gefs_without_herbie(rundt, fcst_hr, [field.upper()], outdir, backend_kwargs, require_all_matches)

# Setup plot configuration and projection
def setup_plot(extent):
    fig = plt.figure(figsize=[10, 10])
    datacrs = ccrs.PlateCarree()
    plotcrs = ccrs.NorthPolarStereo(true_scale_latitude=60, central_longitude=202.993)
    ax = plt.axes(projection=plotcrs)
    ax.set_extent(extent, datacrs)
    ax.add_feature(cfeature.STATES, edgecolor='black', linewidth=1)
    ax.add_feature(cfeature.COASTLINE.with_scale('50m'))
    return fig, ax, datacrs

# Colormap and range setup
def configure_colormap(colorlist, var):
    cmap = ListedColormap(colorlist)
    contourrange = np.arange(model_cbar_mapping[var][0], model_cbar_mapping[var][1], model_cbar_mapping[var][2])
    cbar_range = np.arange(model_cbar_mapping[var][0], model_cbar_mapping[var][1], model_cbar_mapping[var][3])
    return cmap, contourrange, cbar_range

# Add plot titles and labels
def set_plot_titles(ax, model, runhour, rundate, forecast_hours, valid_times, pos, time_valid):
    title = f"{model} | Init {runhour}Z {datetime.strptime(rundate, '%Y-%m-%d').strftime('%d %b %Y')} | Mean MSLP + Low Locations (hPa)"
    #ax.set_title(title, fontsize=9, weight='bold', loc='left', color='#272727')
    #ax.text(1.00, 1.017, f'Hour: {forecast_hours[model_valid_times[pos]]} | Valid: {time_valid.strftime("%HZ %d %b %Y")}', transform=ax.transAxes, fontsize=9, color='k', weight='semibold')
    ax.set_title(title,fontsize=9, weight='bold', stretch='normal', family='sans-serif', loc='left',color='#272727')
    plt.text(1.00, 1.017, f'Hour: {forecast_hours[valid_times[pos]]} | Valid: {time_valid.strftime("%HZ %d %b %Y")}', horizontalalignment='right',fontsize=9,color='k',transform = ax.transAxes, weight='semibold' )
    #adding copyright infor for the ECMWF data
    if model == "EPS":
      plt.text(1.00,0.01,f"Data Copyright © {datetime.strptime(rundate, '%Y-%m-%d').strftime('%Y')} European Centre for Medium-Range Weather Forecasts (ECMWF)",horizontalalignment='right',color='k',transform = ax.transAxes, fontsize=7, style="italic")


def plot_storm_tracks_dev2(grouped_lows):
    # Set up projections and plot configurations
    fig, ax, datacrs = setup_plot(extent)

    # Define pressure-to-color mapping
    levels = np.arange(910, 1020, 10)
    cmap = plt.cm.jet
    norm = matplotlib.colors.BoundaryNorm(boundaries=levels, ncolors=cmap.N)

    for member, tracks in grouped_lows.items():
        print(f"Plotting for ensemble member: {member}")
        for track_id, track_data in tracks.items():
            pressures = track_data['pressure']
            lats = track_data['lat']
            lons = track_data['lon']

            if len(lats) < 2:
                continue  # Skip tracks too short for arrow direction

            # Get color for each segment
            colors = [cmap(norm(p)) for p in pressures]

            # Plot track with color gradient, linewidth, alpha
            for i in range(len(lats) - 1):
                ax.plot(
                    [lons[i], lons[i+1]],
                    [lats[i], lats[i+1]],
                    color=colors[i],
                    linewidth=1.5,
                    alpha=0.7,
                    transform=datacrs
                )

            # Plot arrows for direction (every other point)
            for i in range(0, len(lats) - 1, 2):
                dx = lons[i+1] - lons[i]
                dy = lats[i+1] - lats[i]
                ax.quiver(
                  np.array([lons[i]]), np.array([lats[i]]),
                  np.array([dx]), np.array([dy]),
                  angles='xy', scale_units='xy', scale=1,
                  width=0.0045, headwidth=4, headlength=5,
                  color=colors[i], transform=datacrs
              )

    # Legend
    legend_elements = [
        Line2D([0], [0], color=cmap(norm(p)), lw=1.5, label=f'{p} hPa')
        for p in np.arange(910, 1020, 10)
    ]
    ax.legend(handles=legend_elements, title="Pressure (hPa)", loc='upper right')

    return fig, ax


def reorder_data(ensemble_data):
    # creating a dictionary of members
    ensemble_dict = {}
    for first_value in ensemble_data.values():
        break
    for member, values in first_value.items():
        ensemble_dict.update({member:{"MinSLP":[], "Lat": [], "Lon":[]}})
    #print(ensemble_dict)
    for i, timestep in ensemble_data.items():
        for member in timestep.keys():
            ensemble_dict[member]["MinSLP"].append(ensemble_data[i][member]["MinSLP"])
            ensemble_dict[member]["Lat"].append(ensemble_data[i][member]["Lat"])
            ensemble_dict[member]["Lon"].append(ensemble_data[i][member]["Lon"])
    return ensemble_dict



# Main plotting function
def plot_data(model, plot_type, ds_list, forecast_hours, valid_times, runstring, runhour, rundate, extent, strength_thresh):
    global filter_tracks
    track_data = {}
    for pos, ds in enumerate(ds_list):
        time_valid = datetime.strptime(runstring, "%Y-%m-%d %H:%M") + timedelta(hours=forecast_hours[pos])

        if model == "EPS":
          # Loop through all datasets in ds_list to find the one with 3 dimensions
          #EPS data comes in a list of datasets, one with all ens members and
          # another representing the mean
          ensemble_ds = None
          mean_ds = None
          for ensds in ds:
              if len(ensds.dims) == 3:
                  ensemble_ds = ensds
              else:
                  mean_ds = ensds
        elif model == "GEFS":
            ensemble_ds = ds
            mean_ds = ds.mean(dim='number')
        # Prepare data dynamically based on model and plot type
        field_data, mean_field_data, lon2d, lat2d = prepare_data_for_plotting(model, plot_type, ensemble_ds, mean_ds)

        if plot_type == "MSLPLowLocs":
            # Set up projections and plot configurations
            fig, ax, datacrs = setup_plot(extent)
            # Colormap and range setup
            cmap, contourrange, cbar_range = configure_colormap(mslcolors, plot_type)
            cf = ax.contourf(lon2d, lat2d, mean_field_data, contourrange, cmap=cmap, transform=datacrs)
            ct = ax.contour(lon2d, lat2d, mean_field_data, contourrange, colors='k', linewidths=0.5, transform=datacrs)

            cbar = plt.colorbar(cf, orientation='horizontal', shrink=0.7, ticks=cbar_range, pad=0, aspect=40, drawedges=True)
            cbar.ax.tick_params(labelsize=10)
            ax.clabel(ct, fmt='%d')

            plot_low_loc(field_data, lat2d, lon2d, ax, datacrs)

            set_plot_titles(ax, model, runhour, rundate, forecast_hours, valid_times, pos, time_valid)
            graphicname = f"{model}_init_{runhour}z_{datetime.strptime(rundate, '%Y-%m-%d').strftime('%Y%m%d')}_F{forecast_hours[pos]}_MSLPLowLocs.png"
            plt.savefig(graphicname, bbox_inches='tight')
            print(f"Done with {graphicname}. Check working directory for the png (folder icon on the left toolbar)")
            plt.close()
        elif plot_type == "StormTracks":
            lowdict = find_low_loc(field_data, lat2d, lon2d)
            #print(f"Low dict is: {lowdict}")
            track_data.update({forecast_hours[pos]:lowdict})
        del ensemble_ds, mean_ds, field_data, mean_field_data, lon2d, lat2d
        gc.collect()

    if plot_type == "StormTracks":
      # restructuring my low track data
      new_track_data = reorder_data(track_data)
      # grouping by lows to make sure we don't mix and match low locations
      if filter_tracks:
        #print("Filtering tracks")
        #print(f"Strength threshold is: {strength_thresh}")
        grouped_lows_unfiltered = group_lows(new_track_data)
        #print(f"Before: {grouped_lows_unfiltered}")
        grouped_lows = filter_lows_by_pressure(grouped_lows_unfiltered, strength_thresh)
        #print(f"After: {grouped_lows}")
      else:
        grouped_lows = group_lows(new_track_data)
      # Plotting storm tracks
      fig, ax = plot_storm_tracks_dev2(grouped_lows)
      #print(grouped_lows)
      title = f"{model} | Init {runhour}Z {datetime.strptime(rundate, '%Y-%m-%d').strftime('%d %b %Y')} | Ensemble Storm Tracks & Strength (hPa)"
      #ax.set_title(title, fontsize=9, weight='bold', loc='left', color='#272727')
      #ax.text(1.00, 1.017, f'Hour: {forecast_hours[model_valid_times[pos]]} | Valid: {time_valid.strftime("%HZ %d %b %Y")}', transform=ax.transAxes, fontsize=9, color='k', weight='semibold')
      ax.set_title(title,fontsize=9, weight='bold', stretch='normal', family='sans-serif', loc='left',color='#272727')
      plt.text(1.00, 1.017, f'Hours: {forecast_hours[valid_times[0]]} - {forecast_hours[valid_times[-1]]} | Valid: {time_valid.strftime("%HZ %d %b %Y")}', horizontalalignment='right',fontsize=9,color='k',transform = ax.transAxes, weight='semibold' )
      #adding copyright infor for the ECMWF data Valid: {time_valid.strftime("%HZ %d %b %Y")}', horizontalalignment='right',fontsize=9,color='k',transform = ax.transAxes, weight='semibold' )
      graphicname = f"{model}_init_{runhour}z_{datetime.strptime(rundate, '%Y-%m-%d').strftime('%Y%m%d')}_StormTracks.png"
      if model == "EPS":
        plt.text(1.00,0.01,f"Data Copyright © {datetime.strptime(rundate, '%Y-%m-%d').strftime('%Y')} European Centre for Medium-Range Weather Forecasts (ECMWF)",horizontalalignment='right',color='k',transform = ax.transAxes, fontsize=7, style="italic")
      #plt.show()
      plt.savefig(graphicname, bbox_inches='tight')
      plt.close()
      #print(dict(islice(new_track_data.items(), 5)))
########################## Main Code ####################################
forecast_hours = list(range(start_hour, end_hour + 1, timestep))

#Getting our model initialization times based on inputs
runhour = str(runhour).zfill(2)
runstring = rundate + " " +runhour + ":00"
run_time = rundate + " " +runhour + "Z"
outdir = "/tmp/gefs_subsets"
model_timesteps = []
model_valid_times = []

for step, fcst_hr in enumerate(forecast_hours):
  #try:
    #with get_model_data(model, runstring, fcst_hr, model_variable_mapping[model][plot_type]) as ds:
  ds = get_model_data(model, runstring, fcst_hr, model_variable_mapping[model][plot_type])
  #print(ds)
  model_timesteps.append(ds)
  model_valid_times.append(step)
  #except Exception as e:
  #  print(e)
  #  continue

#print(model_timesteps)
#print(model_valid_times)
#Plotting data
#if plot_type == "StormTracks":
plot_data(model, plot_type, model_timesteps, forecast_hours, model_valid_times, runstring, runhour, rundate, extent, filter_threshold)

